In [1]:
# Cell 1: Setup and Imports
import os
import re
import json
from pathlib import Path
from collections import defaultdict
import random
import numpy as np

# Set random seed
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"✓ Random seed set to: {RANDOM_SEED}")
print(f"✓ Imports complete")

✓ Random seed set to: 42
✓ Imports complete


In [2]:
# Cell 2: Define Paths and Load Split Info
BASE_DIR = Path('/teamspace/studios/this_studio')
FINAL_EXP_DIR = BASE_DIR / 'final_experiments'
DATA_DIR = BASE_DIR / 'TransMuCoRes' / 'processed_data_full'

# Check if base split exists
split_info_file = FINAL_EXP_DIR / 'dataset_split_info.json'

print("Checking for base split...")
print(f"Looking for: {split_info_file}")
print()

if not split_info_file.exists():
    print("❌ ERROR: Base split not found!")
    print("   Please run 'dataset_split_stratified.ipynb' first.")
else:
    print("✓ Base split found!")
    
    # Load split info
    with open(split_info_file, 'r') as f:
        split_info = json.load(f)
    
    print(f"✓ Loaded split info (seed={split_info['random_seed']})")
    print(f"\nDocument IDs available:")
    print(f"  • BenCoref train: {len(split_info['document_ids']['shared_across_all_experiments']['train_bencoref_41'])} docs")
    print(f"  • TransMuCoRes:   {len(split_info['document_ids']['experiment_1_5_6']['train_transmucores_100'])} docs")
    print(f"  • Dev:            {len(split_info['document_ids']['shared_across_all_experiments']['dev_10'])} docs")
    print(f"  • Test:           {len(split_info['document_ids']['shared_across_all_experiments']['test_71'])} docs")
    
    print(f"\nExperiment status:")
    for exp_name, exp_info in split_info['experiments'].items():
        status = exp_info['status']
        emoji = "✅" if status == "complete" else "⚠️"
        print(f"  {emoji} {exp_name}: {status}")

Checking for base split...
Looking for: /teamspace/studios/this_studio/final_experiments/dataset_split_info.json

✓ Base split found!
✓ Loaded split info (seed=42)

Document IDs available:
  • BenCoref train: 41 docs
  • TransMuCoRes:   100 docs
  • Dev:            10 docs
  • Test:           71 docs

Experiment status:
  ✅ experiment_1: complete
  ✅ experiment_2: complete
  ⚠️ experiment_3: awaiting_augmented_data
  ⚠️ experiment_4: awaiting_augmented_data
  ⚠️ experiment_5: awaiting_augmented_data
  ⚠️ experiment_6: awaiting_augmented_data


In [3]:
# Cell 3: Helper Functions for CoNLL Processing
def extract_documents_from_conll(filepath):
    """Extract individual documents from CoNLL file"""
    documents = []
    current_doc = []
    current_doc_id = None
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            
            if line.startswith('#begin document'):
                match = re.search(r'#begin document \((.+?)\)', line)
                if match:
                    current_doc_id = match.group(1)
                current_doc = [line]
            
            elif line.startswith('#end document'):
                current_doc.append(line)
                if current_doc_id:
                    doc_text = '\n'.join(current_doc)
                    documents.append((current_doc_id, doc_text))
                current_doc = []
                current_doc_id = None
            
            else:
                current_doc.append(line)
    
    return dict(documents)  # Return as {doc_id: doc_text}

def write_conll_file(documents, filepath):
    """Write documents to CoNLL file"""
    with open(filepath, 'w', encoding='utf-8') as f:
        for i, (doc_id, doc_text) in enumerate(documents):
            f.write(doc_text)
            if i < len(documents) - 1:
                f.write('\n\n')
            else:
                f.write('\n')

print("✓ Helper functions defined")
print("  - extract_documents_from_conll()")
print("  - write_conll_file()")

✓ Helper functions defined
  - extract_documents_from_conll()
  - write_conll_file()


In [4]:
# Cell 4: Load Training Documents
print("Loading training documents from Experiments 1 & 2...")
print("="*70)

# Load from existing experiment directories
exp1_train_file = FINAL_EXP_DIR / 'experiment_1_transmucores_bencoref' / 'train.conll'
exp2_train_file = FINAL_EXP_DIR / 'experiment_2_bencoref_only' / 'train.conll'

print(f"\nReading from:")
print(f"  • {exp1_train_file}")
print(f"  • {exp2_train_file}")

# Extract all documents
exp1_docs = extract_documents_from_conll(exp1_train_file)
exp2_docs = extract_documents_from_conll(exp2_train_file)

print(f"\n✓ Loaded {len(exp1_docs)} documents from Experiment 1")
print(f"✓ Loaded {len(exp2_docs)} documents from Experiment 2")

# Identify which are BenCoref vs TransMuCoRes
bencoref_train_docs = exp2_docs  # These are the 41 BenCoref
transmucores_docs = {k: v for k, v in exp1_docs.items() if k not in bencoref_train_docs}

print(f"\n" + "="*70)
print(f"IDENTIFIED:")
print(f"="*70)
print(f"  • {len(bencoref_train_docs)} BenCoref training docs")
print(f"  • {len(transmucores_docs)} TransMuCoRes docs")

# Verify counts
assert len(bencoref_train_docs) == 41, f"Expected 41 BenCoref docs, got {len(bencoref_train_docs)}"
assert len(transmucores_docs) == 100, f"Expected 100 TransMuCoRes docs, got {len(transmucores_docs)}"

print("\n✓ Verification passed!")

# Show sample document IDs
print(f"\nSample BenCoref IDs: {list(bencoref_train_docs.keys())[:3]}")
print(f"Sample TransMuCoRes IDs: {list(transmucores_docs.keys())[:3]}")

Loading training documents from Experiments 1 & 2...

Reading from:
  • /teamspace/studios/this_studio/final_experiments/experiment_1_transmucores_bencoref/train.conll
  • /teamspace/studios/this_studio/final_experiments/experiment_2_bencoref_only/train.conll

✓ Loaded 141 documents from Experiment 1
✓ Loaded 41 documents from Experiment 2

IDENTIFIED:
  • 41 BenCoref training docs
  • 100 TransMuCoRes docs

✓ Verification passed!

Sample BenCoref IDs: ['train_008', 'train_011', 'train_006']
Sample TransMuCoRes IDs: ['105_persuasion_brat_ben_Beng', '1064_the_masque_of_the_red_death_brat_ben_Beng', '110_tess_of_the_durbervilles_a_pure_woman_brat_ben_Beng']


In [5]:
# Cell 5: Function to Extract Text from CoNLL Documents
def extract_sentences_from_conll(doc_text):
    """
    Extract plain text sentences from CoNLL format document.
    Returns list of sentences as strings.
    """
    lines = doc_text.split('\n')
    sentences = []
    current_sentence = []
    
    for line in lines:
        # Skip document markers and empty lines
        if line.startswith('#begin') or line.startswith('#end'):
            continue
        
        if not line.strip():
            # Empty line = sentence boundary
            if current_sentence:
                sentences.append(' '.join(current_sentence))
                current_sentence = []
            continue
        
        # CoNLL format: columns separated by whitespace/tabs
        # Column 3 (index 3) is the word/token
        parts = line.split()
        if len(parts) >= 4:
            token = parts[3]
            current_sentence.append(token)
    
    # Don't forget last sentence
    if current_sentence:
        sentences.append(' '.join(current_sentence))
    
    return sentences

# Test on a sample document
print("Testing text extraction on a sample BenCoref document...")
print("="*70)

sample_doc_id = list(bencoref_train_docs.keys())[0]
sample_doc_text = bencoref_train_docs[sample_doc_id]

extracted_sentences = extract_sentences_from_conll(sample_doc_text)

print(f"\nDocument ID: {sample_doc_id}")
print(f"Number of sentences extracted: {len(extracted_sentences)}")
print(f"\nFirst 3 sentences:")
for i, sent in enumerate(extracted_sentences[:3]):
    print(f"  [{i+1}] {sent}")

print("\n" + "="*70)
print("✓ Text extraction working!")

Testing text extraction on a sample BenCoref document...

Document ID: train_008
Number of sentences extracted: 26

First 3 sentences:
  [1] দুদু মিয়া ( ১৮১৯ - ১৮৬২ ) হাজী শরীয়তউল্লাহর একমাত্র পুত্র ।
  [2] তিনি ১৮১৯ সালে মাদারীপুর জেলার শ্যামাইল গ্রামে জন্মগ্রহণ করেন ।
  [3] তাঁর আসল নাম মুহসীনউদ্দীন ।

✓ Text extraction working!


In [11]:
# Cell 6: Install Required Libraries for Augmentation (UPDATED)
print("Installing augmentation libraries...")
print("="*70)
print("\nThis will install:")
print("  • googletrans (for back-translation)")
print("  • transformers (for BanglaBERT paraphrasing)")
print("\nThis may take 1-2 minutes...\n")

import sys
import subprocess

# Install required packages
packages = [
    'googletrans==4.0.0rc1',  # Specific version that works
    'transformers',
    'torch'
]

for package in packages:
    print(f"Installing {package}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
    print(f"  ✓ {package} installed")

print("\n" + "="*70)
print("✓ All libraries installed!")

# Import and verify
from googletrans import Translator
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

print(f"\n✓ googletrans imported")
print(f"✓ Transformers imported")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

Installing augmentation libraries...

This will install:
  • googletrans (for back-translation)
  • transformers (for BanglaBERT paraphrasing)

This may take 1-2 minutes...

Installing googletrans==4.0.0rc1...
  ✓ googletrans==4.0.0rc1 installed
Installing transformers...
  ✓ transformers installed
Installing torch...
  ✓ torch installed

✓ All libraries installed!

✓ googletrans imported
✓ Transformers imported
✓ PyTorch version: 2.9.0+cu128
✓ CUDA available: True
✓ GPU: NVIDIA L4


In [12]:
# Cell 7: Load Augmentation Models (SIMPLIFIED)
print("Loading augmentation models...")
print("="*70)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

# ============================================================
# 1. BACK-TRANSLATION: Google Translate
# ============================================================
print("\n📥 Initializing Google Translate for back-translation...")
print("   Strategy: Bengali → English → Bengali")

translator = Translator()
print("   ✓ Google Translate initialized")

# ============================================================
# 2. PARAPHRASING: BanglaBERT MLM
# ============================================================
print("\n📥 Loading BanglaBERT for paraphrasing...")
para_model_name = "csebuetnlp/banglabert"
para_tokenizer = AutoTokenizer.from_pretrained(para_model_name)
para_model = AutoModelForMaskedLM.from_pretrained(para_model_name).to(device)
para_model.eval()
print("   ✓ BanglaBERT loaded")

print("\n" + "="*70)
print("✅ ALL MODELS LOADED SUCCESSFULLY!")
print("="*70)
print("\nModels ready:")
print("  ✓ Back-translation: Google Translate (bn↔en)")
print("  ✓ Paraphrasing: BanglaBERT MLM")
print(f"  ✓ Device: {device}")

Loading augmentation models...

Using device: cuda

📥 Initializing Google Translate for back-translation...
   Strategy: Bengali → English → Bengali
   ✓ Google Translate initialized

📥 Loading BanglaBERT for paraphrasing...


Some weights of ElectraForMaskedLM were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['generator_lm_head.bias', 'generator_predictions.LayerNorm.bias', 'generator_predictions.LayerNorm.weight', 'generator_predictions.dense.bias', 'generator_predictions.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   ✓ BanglaBERT loaded

✅ ALL MODELS LOADED SUCCESSFULLY!

Models ready:
  ✓ Back-translation: Google Translate (bn↔en)
  ✓ Paraphrasing: BanglaBERT MLM
  ✓ Device: cuda


In [13]:
# Cell 8: Augmentation Functions (UPDATED)
print("Creating augmentation functions...")
print("="*70)

import time

# ============================================================
# BACK-TRANSLATION FUNCTION (Google Translate)
# ============================================================
def back_translate_sentence(sentence, max_retries=3):
    """
    Back-translate Bengali sentence using Google Translate: bn → en → bn
    Includes rate limiting and retry logic.
    """
    for attempt in range(max_retries):
        try:
            # Step 1: Bengali → English
            english = translator.translate(sentence, src='bn', dest='en').text
            time.sleep(0.3)  # Rate limiting
            
            # Step 2: English → Bengali
            back_translated = translator.translate(english, src='en', dest='bn').text
            time.sleep(0.3)  # Rate limiting
            
            return back_translated
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1)  # Wait before retry
                continue
            else:
                print(f"  ⚠️  BT failed after {max_retries} attempts: {str(e)[:50]}")
                return sentence  # Return original if all retries fail

# ============================================================
# PARAPHRASING FUNCTION (Same as before)
# ============================================================
def paraphrase_sentence(sentence, mask_prob=0.15, top_k=10, max_length=512):
    """
    Paraphrase Bengali sentence using BanglaBERT MLM
    """
    try:
        # Tokenize
        tokens = para_tokenizer.tokenize(sentence)
        
        if len(tokens) == 0:
            return sentence
        
        # Randomly mask some tokens (excluding special tokens)
        n_mask = max(1, int(len(tokens) * mask_prob))
        maskable_indices = [i for i, tok in enumerate(tokens) if not tok.startswith('[')]
        
        if len(maskable_indices) == 0:
            return sentence
        
        mask_indices = random.sample(maskable_indices, min(n_mask, len(maskable_indices)))
        
        # Create masked version
        masked_tokens = tokens.copy()
        for idx in mask_indices:
            masked_tokens[idx] = para_tokenizer.mask_token
        
        # Convert to text and encode
        masked_text = para_tokenizer.convert_tokens_to_string(masked_tokens)
        inputs = para_tokenizer(masked_text, return_tensors="pt", 
                                max_length=max_length, truncation=True).to(device)
        
        # Predict masked tokens
        with torch.no_grad():
            outputs = para_model(**inputs)
        
        predictions = outputs.logits
        
        # Replace masked tokens with predictions
        result_tokens = tokens.copy()
        input_ids = inputs['input_ids'][0].tolist()
        
        for i, token_id in enumerate(input_ids):
            if token_id == para_tokenizer.mask_token_id:
                # Get top-k predictions
                top_k_ids = torch.topk(predictions[0, i], top_k).indices.tolist()
                # Pick randomly from top-k for diversity
                predicted_id = random.choice(top_k_ids)
                predicted_token = para_tokenizer.convert_ids_to_tokens([predicted_id])[0]
                
                # Find corresponding position in original tokens
                for j, tok in enumerate(result_tokens):
                    if tok == para_tokenizer.mask_token:
                        result_tokens[j] = predicted_token
                        break
        
        paraphrased = para_tokenizer.convert_tokens_to_string(result_tokens)
        return paraphrased
    
    except Exception as e:
        print(f"  ⚠️  Para failed: {str(e)[:50]}")
        return sentence

print("✓ back_translate_sentence() defined (Google Translate)")
print("✓ paraphrase_sentence() defined (BanglaBERT)")

# ============================================================
# TEST ON SAMPLE SENTENCES
# ============================================================
print("\n" + "="*70)
print("TESTING AUGMENTATION ON SAMPLE SENTENCES")
print("="*70)

# Get first sentence from our sample doc
test_sentence = extracted_sentences[0]
print(f"\n📝 Original:")
print(f"   {test_sentence}")

print(f"\n🔄 Back-translating (Google Translate)...")
bt_result = back_translate_sentence(test_sentence)
print(f"   {bt_result}")

print(f"\n🎭 Paraphrasing (BanglaBERT)...")
para_result = paraphrase_sentence(test_sentence, mask_prob=0.15)
print(f"   {para_result}")

print("\n" + "="*70)
print("✅ AUGMENTATION FUNCTIONS WORKING!")
print("="*70)

Creating augmentation functions...
✓ back_translate_sentence() defined (Google Translate)
✓ paraphrase_sentence() defined (BanglaBERT)

TESTING AUGMENTATION ON SAMPLE SENTENCES

📝 Original:
   দুদু মিয়া ( ১৮১৯ - ১৮৬২ ) হাজী শরীয়তউল্লাহর একমাত্র পুত্র ।

🔄 Back-translating (Google Translate)...
   দুদু মিয়া (১৮১৯-১৮৬২) ছিলেন হাজী শরীয়তুল্লাহর একমাত্র পুত্র।

🎭 Paraphrasing (BanglaBERT)...
   দুদু মিয়া ( ১৮১৯ - ১৮৬২ ) হাজী শরীয়তউল্লাহর একমাত্র পুত্র ।

✅ AUGMENTATION FUNCTIONS WORKING!


In [15]:
# Cell 9: Document-Level Augmentation Functions (FIXED)
print("Creating document-level augmentation functions...")
print("="*70)

def rebuild_conll_with_new_text(original_conll, new_sentences):
    """
    Rebuild CoNLL document with new sentence text.
    Strategy: Replace entire sentences while keeping CoNLL structure.
    """
    lines = original_conll.split('\n')
    result_lines = []
    sent_idx = 0
    token_idx_in_sent = 0
    
    # Get tokens for current sentence
    current_sent_tokens = new_sentences[sent_idx].split() if sent_idx < len(new_sentences) else []
    
    for line in lines:
        # Keep document markers as-is
        if line.startswith('#begin') or line.startswith('#end'):
            result_lines.append(line)
            continue
        
        # Empty line = sentence boundary
        if not line.strip():
            result_lines.append(line)
            
            # Move to next sentence
            if sent_idx < len(new_sentences) - 1:
                sent_idx += 1
                token_idx_in_sent = 0
                current_sent_tokens = new_sentences[sent_idx].split()
            continue
        
        # Parse CoNLL line (token line)
        parts = line.split('\t') if '\t' in line else line.split()
        
        if len(parts) < 4:
            # Malformed line, keep as-is
            result_lines.append(line)
            continue
        
        # Replace word column (index 3) with new token
        if token_idx_in_sent < len(current_sent_tokens):
            parts[3] = current_sent_tokens[token_idx_in_sent]
            token_idx_in_sent += 1
            result_lines.append('\t'.join(parts) if '\t' in line else ' '.join(parts))
        else:
            # No more tokens in new sentence - skip this line
            # This handles cases where BT produces shorter sentences
            continue
    
    return '\n'.join(result_lines)

def augment_document_bt(doc_id, doc_text):
    """Create back-translated version of a CoNLL document"""
    # Extract sentences
    sentences = extract_sentences_from_conll(doc_text)
    
    # Back-translate each sentence
    bt_sentences = []
    for i, sent in enumerate(sentences):
        if i % 10 == 0 and i > 0:
            print(f"    BT progress: {i}/{len(sentences)} sentences", end='\r')
        bt_sent = back_translate_sentence(sent)
        bt_sentences.append(bt_sent)
    
    # Rebuild CoNLL
    bt_doc_text = rebuild_conll_with_new_text(doc_text, bt_sentences)
    
    # Update document ID in the CoNLL markers
    bt_doc_id = doc_id + '_bt'
    bt_doc_text = bt_doc_text.replace(f'({doc_id})', f'({bt_doc_id})')
    
    return bt_doc_id, bt_doc_text

def augment_document_para(doc_id, doc_text, mask_prob=0.15):
    """Create paraphrased version of a CoNLL document"""
    # Extract sentences
    sentences = extract_sentences_from_conll(doc_text)
    
    # Paraphrase each sentence
    para_sentences = []
    for i, sent in enumerate(sentences):
        if i % 10 == 0 and i > 0:
            print(f"    Para progress: {i}/{len(sentences)} sentences", end='\r')
        para_sent = paraphrase_sentence(sent, mask_prob=mask_prob)
        para_sentences.append(para_sent)
    
    # Rebuild CoNLL
    para_doc_text = rebuild_conll_with_new_text(doc_text, para_sentences)
    
    # Update document ID
    para_doc_id = doc_id + '_para'
    para_doc_text = para_doc_text.replace(f'({doc_id})', f'({para_doc_id})')
    
    return para_doc_id, para_doc_text

print("✓ rebuild_conll_with_new_text() defined (FIXED)")
print("✓ augment_document_bt() defined")
print("✓ augment_document_para() defined")

# ============================================================
# TEST ON ONE DOCUMENT
# ============================================================
print("\n" + "="*70)
print("TESTING DOCUMENT-LEVEL AUGMENTATION")
print("="*70)

print(f"\n📄 Testing on document: {sample_doc_id}")
print(f"   Original sentences: {len(extracted_sentences)}")

print(f"\n🔄 Back-translating entire document...")
bt_doc_id, bt_doc_text = augment_document_bt(sample_doc_id, sample_doc_text)
bt_sentences = extract_sentences_from_conll(bt_doc_text)
print(f"   ✓ BT document created: {bt_doc_id}")
print(f"   ✓ BT sentences: {len(bt_sentences)}")

print(f"\n🎭 Paraphrasing entire document...")
para_doc_id, para_doc_text = augment_document_para(sample_doc_id, sample_doc_text, mask_prob=0.2)
para_sentences = extract_sentences_from_conll(para_doc_text)
print(f"   ✓ Para document created: {para_doc_id}")
print(f"   ✓ Para sentences: {len(para_sentences)}")

# Show comparison of first 3 sentences
print(f"\n📊 Comparison (first 3 sentences):")
for i in range(min(3, len(extracted_sentences))):
    print(f"\n  Sentence {i+1}:")
    print(f"    Original: {extracted_sentences[i]}")
    print(f"    BT:       {bt_sentences[i]}")
    print(f"    Para:     {para_sentences[i]}")

print("\n" + "="*70)
print("✅ DOCUMENT AUGMENTATION WORKING!")
print("="*70)

Creating document-level augmentation functions...
✓ rebuild_conll_with_new_text() defined (FIXED)
✓ augment_document_bt() defined
✓ augment_document_para() defined

TESTING DOCUMENT-LEVEL AUGMENTATION

📄 Testing on document: train_008
   Original sentences: 26

🔄 Back-translating entire document...
   ✓ BT document created: train_008_bt
   ✓ BT sentences: 26

🎭 Paraphrasing entire document...
   ✓ Para document created: train_008_para
   ✓ Para sentences: 26

📊 Comparison (first 3 sentences):

  Sentence 1:
    Original: দুদু মিয়া ( ১৮১৯ - ১৮৬২ ) হাজী শরীয়তউল্লাহর একমাত্র পুত্র ।
    BT:       দুদু মিয়া (১৮১৯-১৮৬২) ছিলেন হাজী শরীয়তুল্লাহর একমাত্র পুত্র।
    Para:     দুদু মিয়া ( ১৮১৯ - ১৮৬২ ) হাজী শরীয়তউল্লাহর একমাত্র পুত্র ।

  Sentence 2:
    Original: তিনি ১৮১৯ সালে মাদারীপুর জেলার শ্যামাইল গ্রামে জন্মগ্রহণ করেন ।
    BT:       তিনি 1819 সালে মাদারীপুর জেলার শামাইল গ্রামে জন্মগ্রহণ করেন।
    Para:     তিনি ১৮১৯ সালে মাদারীপুর জেলার শ্যামাইল গ্রামে জন্মগ্রহণ করেন ।

  Sentence 

In [16]:
# Cell 10: Create ALL Augmented Documents (with Google Translate)
print("="*70)
print("CREATING AUGMENTED VERSIONS OF ALL TRAINING DOCUMENTS")
print("="*70)
print("\n⏱️  Estimated time: 45-90 minutes total")
print("    (Google Translate has rate limits, so we go slower)")
print()

import time
from tqdm.auto import tqdm

# ============================================================
# AUGMENT BENCOREF TRAINING DOCS (41 docs)
# ============================================================
print("\n" + "="*70)
print("STEP 1/4: Back-translating 41 BenCoref documents (Google Translate)")
print("="*70)

bencoref_bt = {}
start_time = time.time()

for i, (doc_id, doc_text) in enumerate(tqdm(bencoref_train_docs.items(), desc="BT BenCoref")):
    bt_id, bt_text = augment_document_bt(doc_id, doc_text)
    bencoref_bt[bt_id] = bt_text
    
    # Extra delay every 10 documents to avoid rate limiting
    if (i + 1) % 10 == 0:
        time.sleep(2)

elapsed = time.time() - start_time
print(f"\n✓ Created {len(bencoref_bt)} back-translated BenCoref docs in {elapsed/60:.1f} minutes")

# ============================================================
print("\n" + "="*70)
print("STEP 2/4: Paraphrasing 41 BenCoref documents")
print("="*70)

bencoref_para = {}
start_time = time.time()

for doc_id, doc_text in tqdm(bencoref_train_docs.items(), desc="Para BenCoref"):
    para_id, para_text = augment_document_para(doc_id, doc_text, mask_prob=0.2)
    bencoref_para[para_id] = para_text

elapsed = time.time() - start_time
print(f"\n✓ Created {len(bencoref_para)} paraphrased BenCoref docs in {elapsed/60:.1f} minutes")

# ============================================================
print("\n" + "="*70)
print("STEP 3/4: Back-translating 100 TransMuCoRes documents (Google Translate)")
print("="*70)

transmucores_bt = {}
start_time = time.time()

for i, (doc_id, doc_text) in enumerate(tqdm(transmucores_docs.items(), desc="BT TransMuCoRes")):
    bt_id, bt_text = augment_document_bt(doc_id, doc_text)
    transmucores_bt[bt_id] = bt_text
    
    # Extra delay every 10 documents
    if (i + 1) % 10 == 0:
        time.sleep(2)

elapsed = time.time() - start_time
print(f"\n✓ Created {len(transmucores_bt)} back-translated TransMuCoRes docs in {elapsed/60:.1f} minutes")

# ============================================================
print("\n" + "="*70)
print("STEP 4/4: Paraphrasing 100 TransMuCoRes documents")
print("="*70)

transmucores_para = {}
start_time = time.time()

for doc_id, doc_text in tqdm(transmucores_docs.items(), desc="Para TransMuCoRes"):
    para_id, para_text = augment_document_para(doc_id, doc_text, mask_prob=0.2)
    transmucores_para[para_id] = para_text

elapsed = time.time() - start_time
print(f"\n✓ Created {len(transmucores_para)} paraphrased TransMuCoRes docs in {elapsed/60:.1f} minutes")

# ============================================================
print("\n" + "="*70)
print("✅ ALL AUGMENTATION COMPLETE!")
print("="*70)
print(f"\nAugmented documents created:")
print(f"  • BenCoref BT:       {len(bencoref_bt)} docs")
print(f"  • BenCoref Para:     {len(bencoref_para)} docs")
print(f"  • TransMuCoRes BT:   {len(transmucores_bt)} docs")
print(f"  • TransMuCoRes Para: {len(transmucores_para)} docs")
print(f"\nTotal: {len(bencoref_bt) + len(bencoref_para) + len(transmucores_bt) + len(transmucores_para)} augmented docs")

CREATING AUGMENTED VERSIONS OF ALL TRAINING DOCUMENTS

⏱️  Estimated time: 45-90 minutes total
    (Google Translate has rate limits, so we go slower)


STEP 1/4: Back-translating 41 BenCoref documents (Google Translate)


BT BenCoref:   0%|          | 0/41 [00:00<?, ?it/s]

    BT progress: 40/46 sentencess
✓ Created 41 back-translated BenCoref docs in 41.0 minutes

STEP 2/4: Paraphrasing 41 BenCoref documents


Para BenCoref:   0%|          | 0/41 [00:00<?, ?it/s]

    Para progress: 40/46 sentencess
✓ Created 41 paraphrased BenCoref docs in 0.5 minutes

STEP 3/4: Back-translating 100 TransMuCoRes documents (Google Translate)


BT TransMuCoRes:   0%|          | 0/100 [00:00<?, ?it/s]

    BT progress: 60/61 sentenceses
✓ Created 100 back-translated TransMuCoRes docs in 281.5 minutes

STEP 4/4: Paraphrasing 100 TransMuCoRes documents


Para TransMuCoRes:   0%|          | 0/100 [00:00<?, ?it/s]

    Para progress: 60/61 sentenceses
✓ Created 100 paraphrased TransMuCoRes docs in 1.9 minutes

✅ ALL AUGMENTATION COMPLETE!

Augmented documents created:
  • BenCoref BT:       41 docs
  • BenCoref Para:     41 docs
  • TransMuCoRes BT:   100 docs
  • TransMuCoRes Para: 100 docs

Total: 282 augmented docs


In [1]:
# Cell 11: Assemble All 6 Experiments (STANDALONE VERSION)
print("="*70)
print("ASSEMBLING ALL 6 EXPERIMENTS (Standalone)")
print("="*70)
print("\nThis cell loads data from disk and doesn't depend on previous cells.")
print()

import os
import re
import json
from pathlib import Path
from collections import defaultdict
import random
import shutil

# Set random seed
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Define paths
BASE_DIR = Path('/teamspace/studios/this_studio')
FINAL_EXP_DIR = BASE_DIR / 'final_experiments'

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def extract_documents_from_conll(filepath):
    """Extract individual documents from CoNLL file"""
    documents = []
    current_doc = []
    current_doc_id = None
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            
            if line.startswith('#begin document'):
                match = re.search(r'#begin document \((.+?)\)', line)
                if match:
                    current_doc_id = match.group(1)
                current_doc = [line]
            
            elif line.startswith('#end document'):
                current_doc.append(line)
                if current_doc_id:
                    doc_text = '\n'.join(current_doc)
                    documents.append((current_doc_id, doc_text))
                current_doc = []
                current_doc_id = None
            
            else:
                current_doc.append(line)
    
    return documents

def write_conll_file(documents, filepath):
    """Write documents to CoNLL file"""
    with open(filepath, 'w', encoding='utf-8') as f:
        for i, (doc_id, doc_text) in enumerate(documents):
            f.write(doc_text)
            if i < len(documents) - 1:
                f.write('\n\n')
            else:
                f.write('\n')

# ============================================================
# CHECK IF AUGMENTED DATA EXISTS
# ============================================================
print("Checking for augmented data...")

# Check if we have the variables in memory first
try:
    # Try to use in-memory variables if Cell 10 was just run
    _ = bencoref_bt
    _ = bencoref_para
    _ = transmucores_bt
    _ = transmucores_para
    print("✓ Using augmented data from memory (Cell 10 was run)")
    use_memory = True
except NameError:
    print("⚠️  Augmented data not in memory")
    print("   Checking for saved augmented files...")
    use_memory = False

if not use_memory:
    # Check if augmented experiments already exist
    exp3_dir = FINAL_EXP_DIR / 'experiment_3_bencoref_backtranslated'
    exp4_dir = FINAL_EXP_DIR / 'experiment_4_bencoref_paraphrased'
    exp5_dir = FINAL_EXP_DIR / 'experiment_5_full_backtranslated'
    exp6_dir = FINAL_EXP_DIR / 'experiment_6_full_paraphrased'
    
    if all([exp3_dir.exists(), exp4_dir.exists(), exp5_dir.exists(), exp6_dir.exists()]):
        print("✓ All augmented experiments already exist!")
        print("   Skipping assembly (experiments 3-6 already created)")
        print("\nExisting experiments:")
        for exp_dir in [exp3_dir, exp4_dir, exp5_dir, exp6_dir]:
            train_file = exp_dir / 'train.conll'
            if train_file.exists():
                docs = extract_documents_from_conll(train_file)
                print(f"  • {exp_dir.name}: {len(docs)} training docs")
        
        print("\n" + "="*70)
        print("✅ EXPERIMENTS 3-6 ALREADY COMPLETE!")
        print("="*70)
        
    else:
        print("❌ ERROR: Augmented data not found!")
        print("\nYou need to run Cell 10 first to create augmented documents.")
        print("Or, if Cell 10 is running, wait for it to complete.")

else:
    # ============================================================
    # LOAD BASE DOCUMENTS
    # ============================================================
    print("\nLoading base documents from Experiments 1 & 2...")
    
    exp1_train_file = FINAL_EXP_DIR / 'experiment_1_transmucores_bencoref' / 'train.conll'
    exp2_train_file = FINAL_EXP_DIR / 'experiment_2_bencoref_only' / 'train.conll'
    
    exp1_docs_list = extract_documents_from_conll(exp1_train_file)
    exp2_docs_list = extract_documents_from_conll(exp2_train_file)
    
    exp1_docs = dict(exp1_docs_list)
    exp2_docs = dict(exp2_docs_list)
    
    bencoref_train_docs = exp2_docs
    transmucores_docs = {k: v for k, v in exp1_docs.items() if k not in bencoref_train_docs}
    
    print(f"  ✓ Loaded {len(bencoref_train_docs)} BenCoref docs")
    print(f"  ✓ Loaded {len(transmucores_docs)} TransMuCoRes docs")
    
    # Prepare combinations
    full_original = list(transmucores_docs.items()) + list(bencoref_train_docs.items())
    
    # ============================================================
    # EXPERIMENT 3: BenCoref + BT BenCoref
    # ============================================================
    print("\n📁 Experiment 3: BenCoref + Back-translated BenCoref")
    exp3_dir = FINAL_EXP_DIR / 'experiment_3_bencoref_backtranslated'
    exp3_dir.mkdir(exist_ok=True)
    
    exp3_train = list(bencoref_train_docs.items()) + list(bencoref_bt.items())
    random.shuffle(exp3_train)
    
    write_conll_file(exp3_train, exp3_dir / 'train.conll')
    shutil.copy(FINAL_EXP_DIR / 'experiment_2_bencoref_only' / 'dev.conll', exp3_dir / 'dev.conll')
    shutil.copy(FINAL_EXP_DIR / 'experiment_2_bencoref_only' / 'test.conll', exp3_dir / 'test.conll')
    
    print(f"   ✓ train.conll: {len(exp3_train)} docs (41 orig + 41 BT)")
    print(f"   ✓ dev.conll:   10 docs (copied)")
    print(f"   ✓ test.conll:  71 docs (copied)")
    
    # ============================================================
    # EXPERIMENT 4: BenCoref + Para BenCoref
    # ============================================================
    print("\n📁 Experiment 4: BenCoref + Paraphrased BenCoref")
    exp4_dir = FINAL_EXP_DIR / 'experiment_4_bencoref_paraphrased'
    exp4_dir.mkdir(exist_ok=True)
    
    exp4_train = list(bencoref_train_docs.items()) + list(bencoref_para.items())
    random.shuffle(exp4_train)
    
    write_conll_file(exp4_train, exp4_dir / 'train.conll')
    shutil.copy(FINAL_EXP_DIR / 'experiment_2_bencoref_only' / 'dev.conll', exp4_dir / 'dev.conll')
    shutil.copy(FINAL_EXP_DIR / 'experiment_2_bencoref_only' / 'test.conll', exp4_dir / 'test.conll')
    
    print(f"   ✓ train.conll: {len(exp4_train)} docs (41 orig + 41 para)")
    print(f"   ✓ dev.conll:   10 docs (copied)")
    print(f"   ✓ test.conll:  71 docs (copied)")
    
    # ============================================================
    # EXPERIMENT 5: Full + BT Full
    # ============================================================
    print("\n📁 Experiment 5: Full Dataset + Back-translated")
    exp5_dir = FINAL_EXP_DIR / 'experiment_5_full_backtranslated'
    exp5_dir.mkdir(exist_ok=True)
    
    full_bt = list(transmucores_bt.items()) + list(bencoref_bt.items())
    exp5_train = full_original + full_bt
    random.shuffle(exp5_train)
    
    write_conll_file(exp5_train, exp5_dir / 'train.conll')
    shutil.copy(FINAL_EXP_DIR / 'experiment_1_transmucores_bencoref' / 'dev.conll', exp5_dir / 'dev.conll')
    shutil.copy(FINAL_EXP_DIR / 'experiment_1_transmucores_bencoref' / 'test.conll', exp5_dir / 'test.conll')
    
    print(f"   ✓ train.conll: {len(exp5_train)} docs (141 orig + 141 BT)")
    print(f"   ✓ dev.conll:   10 docs (copied)")
    print(f"   ✓ test.conll:  71 docs (copied)")
    
    # ============================================================
    # EXPERIMENT 6: Full + Para Full
    # ============================================================
    print("\n📁 Experiment 6: Full Dataset + Paraphrased")
    exp6_dir = FINAL_EXP_DIR / 'experiment_6_full_paraphrased'
    exp6_dir.mkdir(exist_ok=True)
    
    full_para = list(transmucores_para.items()) + list(bencoref_para.items())
    exp6_train = full_original + full_para
    random.shuffle(exp6_train)
    
    write_conll_file(exp6_train, exp6_dir / 'train.conll')
    shutil.copy(FINAL_EXP_DIR / 'experiment_1_transmucores_bencoref' / 'dev.conll', exp6_dir / 'dev.conll')
    shutil.copy(FINAL_EXP_DIR / 'experiment_1_transmucores_bencoref' / 'test.conll', exp6_dir / 'test.conll')
    
    print(f"   ✓ train.conll: {len(exp6_train)} docs (141 orig + 141 para)")
    print(f"   ✓ dev.conll:   10 docs (copied)")
    print(f"   ✓ test.conll:  71 docs (copied)")
    
    # ============================================================
    # SUMMARY
    # ============================================================
    print("\n" + "="*70)
    print("✅ ALL 6 EXPERIMENTS ASSEMBLED!")
    print("="*70)
    
    experiments = [
        ("Experiment 1", "experiment_1_transmucores_bencoref", 141),
        ("Experiment 2", "experiment_2_bencoref_only", 41),
        ("Experiment 3", "experiment_3_bencoref_backtranslated", 82),
        ("Experiment 4", "experiment_4_bencoref_paraphrased", 82),
        ("Experiment 5", "experiment_5_full_backtranslated", 282),
        ("Experiment 6", "experiment_6_full_paraphrased", 282)
    ]
    
    print("\n📊 Final Dataset Structure:")
    print("-" * 70)
    for name, dirname, train_size in experiments:
        print(f"\n{name}: {dirname}/")
        print(f"  • train.conll: {train_size} docs")
        print(f"  • dev.conll:   10 docs")
        print(f"  • test.conll:  71 docs")
    
    print("\n" + "="*70)
    print("KEY POINTS:")
    print("="*70)
    print("✓ All 6 experiments complete")
    print("✓ Same DEV set (10 docs) across ALL experiments")
    print("✓ Same TEST set (71 docs) across ALL experiments")
    print("✓ Augmentation: Google Translate BT + BanglaBERT Para")
    print("✓ Random seed: 42")
    print("="*70)

ASSEMBLING ALL 6 EXPERIMENTS (Standalone)

This cell loads data from disk and doesn't depend on previous cells.

Checking for augmented data...
⚠️  Augmented data not in memory
   Checking for saved augmented files...
✓ All augmented experiments already exist!
   Skipping assembly (experiments 3-6 already created)

Existing experiments:
  • experiment_3_bencoref_backtranslated: 82 training docs
  • experiment_4_bencoref_paraphrased: 82 training docs
  • experiment_5_full_backtranslated: 282 training docs
  • experiment_6_full_paraphrased: 282 training docs

✅ EXPERIMENTS 3-6 ALREADY COMPLETE!


In [3]:
# Cell 12: Update Metadata and Final Summary (DETAILED VERSION)
print("="*70)
print("UPDATING METADATA & CREATING FINAL SUMMARY")
print("="*70)

import json
from pathlib import Path
from datetime import datetime
from collections import defaultdict

BASE_DIR = Path('/teamspace/studios/this_studio')
FINAL_EXP_DIR = BASE_DIR / 'final_experiments'
split_info_file = FINAL_EXP_DIR / 'dataset_split_info.json'

# Load existing split info
with open(split_info_file, 'r') as f:
    split_info = json.load(f)

# Update experiment statuses
split_info['experiments']['experiment_3']['status'] = 'complete'
split_info['experiments']['experiment_4']['status'] = 'complete'
split_info['experiments']['experiment_5']['status'] = 'complete'
split_info['experiments']['experiment_6']['status'] = 'complete'

# Update summary
split_info['summary']['completed'] = 6
split_info['summary']['awaiting_augmentation'] = 0

# Add augmentation details
split_info['augmentation'] = {
    'creation_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'methods': {
        'back_translation': {
            'tool': 'Google Translate (googletrans 4.0.0rc1)',
            'strategy': 'Bengali → English → Bengali',
            'notes': 'Rate-limited to avoid blocking. ~0.6s per sentence.'
        },
        'paraphrasing': {
            'tool': 'BanglaBERT MLM (csebuetnlp/banglabert)',
            'strategy': 'Mask 20% of tokens, predict with top-k sampling',
            'mask_probability': 0.2,
            'top_k': 10
        }
    },
    'statistics': {
        'bencoref_bt': 41,
        'bencoref_para': 41,
        'transmucores_bt': 100,
        'transmucores_para': 100,
        'total_augmented': 282
    },
    'time_taken': {
        'bencoref_bt': '~41 minutes',
        'bencoref_para': '~0.5 minutes',
        'transmucores_bt': '~282 minutes',
        'transmucores_para': '~2 minutes',
        'total': '~5.5 hours'
    }
}

# Save updated split info
with open(split_info_file, 'w') as f:
    json.dump(split_info, f, indent=2, ensure_ascii=False)

print("✓ dataset_split_info.json updated")

# ============================================================
# HELPER: Extract document IDs by domain
# ============================================================
def get_domain_from_doc_id(doc_id):
    """Infer domain from document ID"""
    # BenCoref docs: train_XXX format
    # TransMuCoRes docs: longer format with domain info
    
    if doc_id.startswith('train_'):
        # Need to check against known domain splits
        return 'bencoref'
    else:
        return 'transmucores'

def count_domains_in_file(filepath, split_info):
    """Count documents by domain using split_info"""
    with open(filepath, 'r') as f:
        content = f.read()
    
    # Extract all document IDs
    import re
    doc_ids = re.findall(r'#begin document \((.+?)\)', content)
    
    # Count by domain using split_info
    domain_counts = defaultdict(int)
    
    # Get domain info from split_info
    bencoref_strat = split_info.get('bencoref_stratification', {})
    
    for doc_id in doc_ids:
        # Check if it's in any domain's doc list
        found = False
        for domain, info in bencoref_strat.items():
            if doc_id in info.get('train_ids', []) or \
               doc_id in info.get('dev_ids', []) or \
               doc_id in info.get('test_ids', []):
                domain_counts[domain] += 1
                found = True
                break
        
        if not found:
            # Must be TransMuCoRes
            domain_counts['transmucores'] += 1
    
    return dict(domain_counts)

# ============================================================
# ANALYZE EACH EXPERIMENT
# ============================================================
print("\n" + "="*70)
print("DETAILED EXPERIMENT ANALYSIS")
print("="*70)

experiments_detail = [
    {
        'name': 'Experiment 1',
        'dir': 'experiment_1_transmucores_bencoref',
        'train_desc': '100 TransMuCoRes + 41 BenCoref',
        'train_size': 141,
        'composition': 'Original data only'
    },
    {
        'name': 'Experiment 2',
        'dir': 'experiment_2_bencoref_only',
        'train_desc': '41 BenCoref',
        'train_size': 41,
        'composition': 'Original BenCoref only'
    },
    {
        'name': 'Experiment 3',
        'dir': 'experiment_3_bencoref_backtranslated',
        'train_desc': '41 BenCoref + 41 BT BenCoref',
        'train_size': 82,
        'composition': 'Original + Back-translated BenCoref'
    },
    {
        'name': 'Experiment 4',
        'dir': 'experiment_4_bencoref_paraphrased',
        'train_desc': '41 BenCoref + 41 Para BenCoref',
        'train_size': 82,
        'composition': 'Original + Paraphrased BenCoref'
    },
    {
        'name': 'Experiment 5',
        'dir': 'experiment_5_full_backtranslated',
        'train_desc': '100 TransMuCoRes + 41 BenCoref + 141 BT',
        'train_size': 282,
        'composition': 'Full dataset + Back-translated'
    },
    {
        'name': 'Experiment 6',
        'dir': 'experiment_6_full_paraphrased',
        'train_desc': '100 TransMuCoRes + 41 BenCoref + 141 Para',
        'train_size': 282,
        'composition': 'Full dataset + Paraphrased'
    }
]

# Get test set domain distribution (same for all experiments)
test_file = FINAL_EXP_DIR / 'experiment_1_transmucores_bencoref' / 'test.conll'
test_domains = count_domains_in_file(test_file, split_info)

# Get dev set domain distribution (same for all experiments)
dev_file = FINAL_EXP_DIR / 'experiment_1_transmucores_bencoref' / 'dev.conll'
dev_domains = count_domains_in_file(dev_file, split_info)

print("\n📊 SHARED DEV SET (10 documents - All BenCoref):")
print("-" * 70)
for domain in sorted(test_domains.keys()):
    if domain in dev_domains:
        count = dev_domains[domain]
        percentage = (count / 10) * 100
        print(f"  {domain.capitalize():12s}: {count:2d} docs ({percentage:5.1f}%)")

print("\n📊 SHARED TEST SET (71 documents - All BenCoref):")
print("-" * 70)
for domain in sorted(test_domains.keys()):
    count = test_domains[domain]
    percentage = (count / 71) * 100
    print(f"  {domain.capitalize():12s}: {count:2d} docs ({percentage:5.1f}%)")

print("\n" + "="*70)
print("DETAILED EXPERIMENT BREAKDOWN")
print("="*70)

for exp in experiments_detail:
    exp_dir = FINAL_EXP_DIR / exp['dir']
    
    print(f"\n{'─'*70}")
    print(f"📁 {exp['name']}: {exp['dir']}")
    print(f"{'─'*70}")
    
    # Verify files exist
    train_file = exp_dir / 'train.conll'
    if not train_file.exists():
        print("  ❌ Train file not found!")
        continue
    
    # Count actual docs
    with open(train_file, 'r') as f:
        actual_train = f.read().count('#begin document')
    
    print(f"\n  Training Set: {actual_train} documents")
    print(f"    Composition: {exp['train_desc']}")
    print(f"    Type: {exp['composition']}")
    
    # Get train domain distribution
    train_domains = count_domains_in_file(train_file, split_info)
    
    if train_domains:
        print(f"\n    Domain breakdown:")
        for domain in sorted(train_domains.keys()):
            count = train_domains[domain]
            percentage = (count / actual_train) * 100
            print(f"      {domain.capitalize():12s}: {count:3d} docs ({percentage:5.1f}%)")
    
    print(f"\n  Dev Set: 10 documents (All BenCoref)")
    print(f"  Test Set: 71 documents (All BenCoref)")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "="*70)
print("🎉 COMPLETE DATASET CREATION SUMMARY")
print("="*70)

print(f"\n📂 Location: {FINAL_EXP_DIR}")

print("\n" + "="*70)
print("KEY FEATURES:")
print("="*70)
print("✓ Random seed: 42 (fully reproducible)")
print("✓ Stratified splits across 4 domains:")
print(f"    • Biography:   {test_domains.get('biography', 0)} test docs")
print(f"    • Descriptive: {test_domains.get('descriptive', 0)} test docs")
print(f"    • Novel:       {test_domains.get('novel', 0)} test docs")
print(f"    • Story:       {test_domains.get('story', 0)} test docs")
print("✓ Same DEV set (10 BenCoref docs) across ALL 6 experiments")
print("✓ Same TEST set (71 BenCoref docs) across ALL 6 experiments")
print("✓ Augmentation methods:")
print("    • Back-translation: Google Translate (bn→en→bn)")
print("    • Paraphrasing: BanglaBERT MLM (20% masking)")
print("✓ Total augmented documents: 282")
print("✓ Total creation time: ~5.5 hours")

print("\n" + "="*70)
print("RESEARCH QUESTIONS YOU CAN ANSWER:")
print("="*70)
print("1. Effect of more data:")
print("   → Exp 1 (141 docs) vs Exp 2 (41 docs)")
print()
print("2. Effect of back-translation on BenCoref:")
print("   → Exp 3 (41 + 41 BT) vs Exp 2 (41 original)")
print()
print("3. Effect of paraphrasing on BenCoref:")
print("   → Exp 4 (41 + 41 Para) vs Exp 2 (41 original)")
print()
print("4. Effect of back-translation on full dataset:")
print("   → Exp 5 (141 + 141 BT) vs Exp 1 (141 original)")
print()
print("5. Effect of paraphrasing on full dataset:")
print("   → Exp 6 (141 + 141 Para) vs Exp 1 (141 original)")
print()
print("6. Back-translation vs paraphrasing:")
print("   → Exp 3 vs Exp 4 (BenCoref augmentation)")
print("   → Exp 5 vs Exp 6 (Full dataset augmentation)")

print("\n" + "="*70)
print("DATASET SIZE PROGRESSION:")
print("="*70)
print("  Exp 2:   41 docs (Baseline)")
print("  Exp 3:   82 docs (2x with BT)")
print("  Exp 4:   82 docs (2x with Para)")
print("  Exp 1:  141 docs (More data)")
print("  Exp 5:  282 docs (2x with BT)")
print("  Exp 6:  282 docs (2x with Para)")

print("\n" + "="*70)
print("FOR YOUR THESIS - KEY STATISTICS:")
print("="*70)
print(f"• Total original documents: 122 BenCoref + 100 TransMuCoRes")
print(f"• Training documents: 41 BenCoref + 100 TransMuCoRes")
print(f"• Dev documents: 10 BenCoref (stratified)")
print(f"• Test documents: 71 BenCoref (stratified)")
print(f"• Augmented documents: 282 (141 BT + 141 Para)")
print(f"• Domain distribution maintained in dev/test splits")
print(f"• All experiments use identical test set for fair comparison")

print("\n" + "="*70)
print("NEXT STEPS:")
print("="*70)
print("1. ✅ Dataset creation complete")
print("2. TODO: Train models on all 6 experiments")
print("   • BanglaBERT-Large")
print("   • MuRIL-Large")
print("   • XLM-RoBERTa-Large")
print("3. TODO: Tune hyperparameters on dev set (10 docs)")
print("4. TODO: Final evaluation on test set (71 docs)")
print("5. TODO: Statistical significance testing")
print("6. TODO: Write thesis results chapter")

print("\n" + "="*70)
print("✅✅✅ ALL DATASETS READY FOR TRAINING! ✅✅✅")
print("="*70)

print(f"\n📁 All files: {FINAL_EXP_DIR}")
print(f"📄 Metadata: {split_info_file}")
print("\n🚀 You can now start training your models!")
print("🎓 Good luck with your experiments!")

UPDATING METADATA & CREATING FINAL SUMMARY
✓ dataset_split_info.json updated

DETAILED EXPERIMENT ANALYSIS

📊 SHARED DEV SET (10 documents - All BenCoref):
----------------------------------------------------------------------
  Biography   :  1 docs ( 10.0%)
  Descriptive :  3 docs ( 30.0%)
  Novel       :  1 docs ( 10.0%)
  Story       :  5 docs ( 50.0%)

📊 SHARED TEST SET (71 documents - All BenCoref):
----------------------------------------------------------------------
  Biography   : 10 docs ( 14.1%)
  Descriptive : 21 docs ( 29.6%)
  Novel       :  8 docs ( 11.3%)
  Story       : 32 docs ( 45.1%)

DETAILED EXPERIMENT BREAKDOWN

──────────────────────────────────────────────────────────────────────
📁 Experiment 1: experiment_1_transmucores_bencoref
──────────────────────────────────────────────────────────────────────

  Training Set: 141 documents
    Composition: 100 TransMuCoRes + 41 BenCoref
    Type: Original data only

    Domain breakdown:
      Biography   :   6 docs (  